# Analysis 1A extension: Drug–dose–duration training exposure fraction


        This notebook quantifies, for each held-out cell and split, the fraction of all
        training perturbation rows matching at least one held-out-cell drug–dose–duration
        condition. Drug and duration must match; dose is matched within ±5% of the held-out dose.

**Audited revision v2:** fixes pandas underscore-column tuple handling, validates every matching step, and exports the exact matched training rows for each held-out cell.

**Optimized revision v3:** matching is indexed by exact `(pert_id, duration)` strata, so each held-out condition is compared only with relevant training rows. The potentially very large matched-row audit table is disabled by default.

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import scipy
from scipy.stats import spearmanr
import scanpy as sc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 1. Configuration

In [ ]:
PROJECT_ROOT = Path("~/DEPICT")
MAIN_WORK_DIR = PROJECT_ROOT / "Code/downstream_analysis_code/TransferabilityUnseenCell"

CONFIG = {
    "adata_path": PROJECT_ROOT / "Data/FinalData/adataAfterClean.h5ad",
    "prediction_root": MAIN_WORK_DIR / "predicted_dge",
    "a1_primary_metrics_path": (
        MAIN_WORK_DIR / "a1_ood_similarity"
        / "tables" / "a1_primary_cell_level_metrics.csv"
    ),
    "output_dir": MAIN_WORK_DIR / "a1_condition_training_exposure",
    "cell_splits": [f"cell_split{i}" for i in range(1, 6)],
    "drug_id_column": "pert_id",
    "drug_name_column": "pert_iname",
    "dose_column": "dose",
    "time_column": "pert_time",
    "cell_column": "cell_id",
    "control_column": "control",
    "dose_tolerance_fraction": 0.05,
    # Exact matched-row audit can be very large and is not needed for plots.
    "save_matched_training_rows_audit": False,
    "dose_round_decimals": 6,
    "time_round_decimals": 6,
    "n_bootstrap": 2000,
    "random_seed": 66,
    "epsilon": 1e-8,
}

OUT = Path(CONFIG["output_dir"])
TABLE_DIR = OUT / "tables"
MANIFEST_DIR = OUT / "manifests"
for directory in (OUT, TABLE_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG

## 2. Statistical and export helpers

In [ ]:
def require(condition: bool, message: str) -> None:
    if not condition:
        raise RuntimeError(message)


def save_table(df: pd.DataFrame, stem: str, index: bool = False) -> None:
    df.to_csv(TABLE_DIR / f"{stem}.csv", index=index)
    try:
        df.to_parquet(TABLE_DIR / f"{stem}.parquet", index=index)
    except Exception as exc:
        print(f"Parquet not written for {stem}: {exc!r}")


def deterministic_seed(*parts) -> int:
    text = "|".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(text).digest()[:8], "little") % (2**32 - 1)


def safe_spearman(x, y, min_n: int = 3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2 or np.unique(y).size < 2:
        return np.nan, np.nan
    result = spearmanr(x, y)
    return float(result.statistic), float(result.pvalue)


def safe_ols(x, y, min_n: int = 3):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if len(x) < min_n or np.unique(x).size < 2:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(x, y, 1)
    fitted = intercept + slope * x
    ss_res = np.sum((y - fitted) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot
    return float(slope), float(intercept), float(r2)


def percentile_ci(values, alpha: float = 0.05):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return tuple(np.quantile(values, [alpha / 2, 1 - alpha / 2]))


def bootstrap_fold_association(df, x_col, y_col, n_bootstrap, seed):
    sub = df[[x_col, y_col]].dropna().reset_index(drop=True)
    x = sub[x_col].to_numpy(float)
    y = sub[y_col].to_numpy(float)
    n = len(sub)

    rho, p = safe_spearman(x, y)
    slope, intercept, ols_r2 = safe_ols(x, y)

    rng = np.random.default_rng(seed)
    boot_rho, boot_slope = [], []
    if n >= 3:
        for _ in range(n_bootstrap):
            ix = rng.integers(0, n, size=n)
            r, _ = safe_spearman(x[ix], y[ix])
            s, _, _ = safe_ols(x[ix], y[ix])
            boot_rho.append(r)
            boot_slope.append(s)

    rho_lo, rho_hi = percentile_ci(boot_rho)
    slope_lo, slope_hi = percentile_ci(boot_slope)

    return {
        "n_cells": n,
        "n_bootstrap_requested": int(n_bootstrap),
        "n_bootstrap_estimable_rho": int(np.isfinite(boot_rho).sum()),
        "n_bootstrap_estimable_slope": int(np.isfinite(boot_slope).sum()),
        "spearman_rho": rho,
        "spearman_p_descriptive": p,
        "spearman_bootstrap_ci_low": rho_lo,
        "spearman_bootstrap_ci_high": rho_hi,
        "ols_slope": slope,
        "ols_intercept": intercept,
        "ols_r2": ols_r2,
        "ols_slope_bootstrap_ci_low": slope_lo,
        "ols_slope_bootstrap_ci_high": slope_hi,
        "association_estimable": bool(np.isfinite(rho)),
    }

## 3. Load and audit perturbation metadata

In [ ]:
adata_path = Path(CONFIG["adata_path"])
require(adata_path.exists(), f"AnnData not found: {adata_path}")

adata = sc.read(adata_path)
require(adata.obs_names.is_unique, "AnnData obs_names must be unique.")

drug_id_col = CONFIG["drug_id_column"]
if drug_id_col not in adata.obs.columns:
    fallback = CONFIG["drug_name_column"]
    require(
        fallback in adata.obs.columns,
        f"Neither {drug_id_col!r} nor fallback {fallback!r} is present in adata.obs.",
    )
    print(f"WARNING: using {fallback!r} because {drug_id_col!r} is unavailable.")
    drug_id_col = fallback

required_obs = [
    drug_id_col,
    CONFIG["dose_column"],
    CONFIG["time_column"],
    CONFIG["cell_column"],
    CONFIG["control_column"],
    *CONFIG["cell_splits"],
]
missing = [c for c in required_obs if c not in adata.obs.columns]
require(not missing, f"AnnData is missing required obs columns: {missing}")

control = pd.to_numeric(adata.obs[CONFIG["control_column"]], errors="coerce")
require(control.notna().all(), "Control column contains missing/non-numeric values.")
require(
    set(control.unique()).issubset({0, 1}),
    f"Unexpected control labels: {sorted(set(control.unique()))}",
)

obs = adata.obs.copy()
obs["_is_perturbation"] = control.to_numpy() == 0
obs["_cell_id"] = obs[CONFIG["cell_column"]].astype("string").str.strip()
obs["_drug_id"] = obs[drug_id_col].astype("string").str.strip()
obs["_dose"] = pd.to_numeric(obs[CONFIG["dose_column"]], errors="coerce").round(
    CONFIG["dose_round_decimals"]
)
obs["_time"] = pd.to_numeric(obs[CONFIG["time_column"]], errors="coerce").round(
    CONFIG["time_round_decimals"]
)

pert_obs = obs.loc[obs["_is_perturbation"]].copy()
valid = (
    pert_obs["_cell_id"].notna()
    & pert_obs["_cell_id"].ne("")
    & pert_obs["_drug_id"].notna()
    & pert_obs["_drug_id"].ne("")
    & pert_obs["_dose"].notna()
    & pert_obs["_time"].notna()
)
require(valid.all(), "Perturbation rows contain missing cell, drug, dose, or duration.")

pert_obs["_cell_id"] = pert_obs["_cell_id"].astype(str)
pert_obs["_drug_id"] = pert_obs["_drug_id"].astype(str)

for split_type in CONFIG["cell_splits"]:
    labels = set(obs[split_type].dropna().astype(str).unique())
    require(
        {"train", "test"}.issubset(labels),
        f"{split_type} must contain both train and test labels. Found: {sorted(labels)}",
    )

print(f"Drug identifier used: {drug_id_col}")
print(f"Perturbation rows: {len(pert_obs):,}")
print(f"Unique cells: {pert_obs['_cell_id'].nunique()}")
print(f"Unique drugs: {pert_obs['_drug_id'].nunique()}")

## 4. Load the original Analysis 1A cell-level performance metrics

In [ ]:
METRIC_COLUMNS = ["delta_pcc", "delta_r2", "dge_mse", "mse_gain_over_naive"]

def decode_h5_strings(values):
    values = np.asarray(values)
    if values.dtype.kind == "S":
        return np.char.decode(values, "utf-8")
    if values.dtype.kind == "O":
        return np.asarray([
            x.decode("utf-8") if isinstance(x, (bytes, bytearray)) else str(x)
            for x in values
        ], dtype=object)
    return values.astype(str)

def safe_pearson_rows(A, B, eps=1e-8, undefined_value=0.0):
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    Ac = A - A.mean(axis=1, keepdims=True)
    Bc = B - B.mean(axis=1, keepdims=True)
    denom = np.sqrt((Ac * Ac).sum(axis=1) * (Bc * Bc).sum(axis=1))
    out = np.full(A.shape[0], float(undefined_value), dtype=float)
    valid = denom > eps
    out[valid] = (Ac[valid] * Bc[valid]).sum(axis=1) / denom[valid]
    return out

def rowwise_r2(y_true, y_pred, eps=1e-12):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    ss_res = ((y_true - y_pred) ** 2).sum(axis=1)
    centered = y_true - y_true.mean(axis=1, keepdims=True)
    ss_tot = (centered ** 2).sum(axis=1)
    out = np.empty(y_true.shape[0], dtype=float)
    nonconstant = ss_tot > eps
    out[nonconstant] = 1.0 - ss_res[nonconstant] / ss_tot[nonconstant]
    out[~nonconstant] = np.where(ss_res[~nonconstant] <= eps, 1.0, 0.0)
    return out

def reconstruct_primary_metrics():
    frames = []
    for split_type in CONFIG["cell_splits"]:
        path = (
            Path(CONFIG["prediction_root"])
            / split_type
            / "predicted_dge_test.h5"
        )
        require(path.exists(), f"Missing prediction HDF5: {path}")

        with h5py.File(path, "r") as h5:
            needed = {"cell_id", "observed_dge", "predicted_dge"}
            missing = needed.difference(h5.keys())
            require(not missing, f"{path} is missing datasets: {sorted(missing)}")

            observed = np.asarray(h5["observed_dge"], dtype=np.float32)
            predicted = np.asarray(h5["predicted_dge"], dtype=np.float32)
            require(
                observed.shape == predicted.shape,
                f"Observed/predicted shape mismatch in {path}.",
            )

            frame = pd.DataFrame({
                "split_type": split_type,
                "cell_id": decode_h5_strings(h5["cell_id"][:]),
                "delta_pcc": safe_pearson_rows(observed, predicted),
                "delta_r2": rowwise_r2(observed, predicted),
                "dge_mse": ((observed - predicted) ** 2).mean(axis=1),
                "naive_dge_mse": (observed ** 2).mean(axis=1),
            })
            frame["mse_gain_over_naive"] = (
                frame["naive_dge_mse"] - frame["dge_mse"]
            )
            frames.append(frame)

    profile_metrics = pd.concat(frames, ignore_index=True)
    cell_metrics = (
        profile_metrics.groupby(["split_type", "cell_id"], as_index=False)[
            METRIC_COLUMNS
        ].mean()
    )
    counts = (
        profile_metrics.groupby(["split_type", "cell_id"])
        .size().rename("n_test_perturbations").reset_index()
    )
    return cell_metrics.merge(
        counts, on=["split_type", "cell_id"], how="left"
    )

metrics_path = Path(CONFIG["a1_primary_metrics_path"])
if metrics_path.exists():
    primary_metrics = pd.read_csv(metrics_path)
    print(f"Loaded original Analysis 1A metrics: {metrics_path}")
else:
    print("Original Analysis 1A metrics not found; reconstructing from HDF5 files.")
    primary_metrics = reconstruct_primary_metrics()

required = ["split_type", "cell_id", *METRIC_COLUMNS]
missing = [c for c in required if c not in primary_metrics.columns]
require(not missing, f"Performance table missing columns: {missing}")

primary_metrics["split_type"] = primary_metrics["split_type"].astype(str)
primary_metrics["cell_id"] = primary_metrics["cell_id"].astype(str)

for split_type in CONFIG["cell_splits"]:
    labels = obs[split_type].astype(str)
    actual_test_cells = set(
        obs.loc[labels == "test", CONFIG["cell_column"]].astype(str).unique()
    )
    metric_cells = set(
        primary_metrics.loc[
            primary_metrics["split_type"] == split_type, "cell_id"
        ].astype(str).unique()
    )
    require(
        metric_cells.issubset(actual_test_cells),
        f"{split_type}: performance table includes non-test cells: "
        f"{sorted(metric_cells.difference(actual_test_cells))}",
    )

save_table(primary_metrics, "cell_level_performance_metrics")
display(primary_metrics.head())

## 5. Compute training exposure fractions

In [ ]:

ANALYSIS_NAME = "drug_dose_duration_training_exposure_fraction"
EXPOSURE_COLUMN = "condition_training_exposure_fraction"
EXPOSURE_DEFINITION = (
    "For each held-out cell and split, the numerator is the number of training "
    "perturbation rows matching at least one held-out-cell condition by identical "
    "pert_id, identical rounded duration, and training dose within ±5% of the "
    "held-out dose. Each training row is counted at most once. The denominator "
    "is the total number of perturbation rows in that split's training set."
)

def build_training_group_index(train_rows: pd.DataFrame):
    """
    Index training-row positions by exact (drug_id, rounded duration).

    This avoids comparing every held-out condition against every training row.
    """
    required = {"_drug_id", "_dose", "_time"}
    require(
        required.issubset(train_rows.columns),
        f"Training rows missing columns: {sorted(required.difference(train_rows.columns))}",
    )

    groups = {}
    grouped = train_rows.groupby(["_drug_id", "_time"], sort=False, dropna=False)
    for (drug_id, duration), group in grouped:
        positions = group.index.to_numpy(dtype=int)
        doses = group["_dose"].to_numpy(dtype=float)
        require(np.isfinite(doses).all(), "Training dose contains non-finite values.")
        groups[(str(drug_id), float(duration))] = (positions, doses)
    return groups


def doses_match_any_test_dose(
    training_doses: np.ndarray,
    test_doses: np.ndarray,
) -> np.ndarray:
    """
    For one drug-duration group, identify training doses that match at least
    one held-out dose under the ±5% rule.

    Zero-dose test conditions match zero exactly (up to numerical storage
    tolerance), because relative tolerance around zero is undefined.
    """
    training_doses = np.asarray(training_doses, dtype=float)
    test_doses = np.asarray(test_doses, dtype=float)

    require(np.isfinite(training_doses).all(), "Non-finite training dose.")
    require(np.isfinite(test_doses).all(), "Non-finite held-out dose.")

    dose_atol = 10.0 ** (-CONFIG["dose_round_decimals"])
    tol_fraction = float(CONFIG["dose_tolerance_fraction"])
    matched = np.zeros(training_doses.shape[0], dtype=bool)

    # The loop is only over unique test doses within one drug-duration group,
    # not over the complete training dataset.
    for test_dose in np.unique(test_doses):
        if np.isclose(test_dose, 0.0, atol=dose_atol, rtol=0.0):
            matched |= np.isclose(
                training_doses,
                0.0,
                atol=dose_atol,
                rtol=0.0,
            )
        else:
            matched |= (
                np.abs(training_doses - test_dose)
                <= tol_fraction * abs(test_dose)
            )
    return matched


def matching_training_mask_indexed(
    train_rows: pd.DataFrame,
    training_group_index: dict,
    test_conditions: pd.DataFrame,
) -> np.ndarray:
    """
    Return a Boolean mask over training rows using indexed drug-duration groups.

    Each training row is counted once at most because matches are accumulated
    into one Boolean mask.
    """
    required = {"_drug_id", "_dose", "_time"}
    require(
        required.issubset(test_conditions.columns),
        f"Test conditions missing columns: {sorted(required.difference(test_conditions.columns))}",
    )

    mask = np.zeros(len(train_rows), dtype=bool)

    # Restrict comparisons to matching drug-duration strata.
    for (drug_id, duration), group in test_conditions.groupby(
        ["_drug_id", "_time"],
        sort=False,
        dropna=False,
    ):
        key = (str(drug_id), float(duration))
        indexed = training_group_index.get(key)
        if indexed is None:
            continue

        training_positions, training_doses = indexed
        test_doses = group["_dose"].to_numpy(dtype=float)
        local_match = doses_match_any_test_dose(training_doses, test_doses)
        mask[training_positions[local_match]] = True

    require(mask.shape == (len(train_rows),), "Unexpected matching-mask shape.")
    return mask


exposure_rows = []
condition_count_rows = []
matched_training_rows_audit = []

for split_type in CONFIG["cell_splits"]:
    split_labels = obs.loc[pert_obs.index, split_type].astype(str)

    # Reset to a 0..n-1 index because the optimized group index stores row positions.
    train_rows = (
        pert_obs.loc[split_labels == "train"]
        .copy()
        .reset_index(drop=False)
        .reset_index(drop=True)
    )
    require(len(train_rows) > 0, f"{split_type}: no training perturbation rows.")

    total_training_rows = len(train_rows)
    training_group_index = build_training_group_index(train_rows)

    test_cells = sorted(
        primary_metrics.loc[
            primary_metrics["split_type"] == split_type,
            "cell_id",
        ].astype(str).unique()
    )
    require(test_cells, f"{split_type}: no held-out cells in performance table.")

    training_condition_counts = (
        train_rows.groupby(
            ["_drug_id", "_dose", "_time"],
            dropna=False,
        )
        .size()
        .rename("n_training_perturbations")
        .reset_index()
    )
    training_condition_counts["split_type"] = split_type
    condition_count_rows.append(training_condition_counts)

    print(
        f"{split_type}: {total_training_rows:,} training perturbations; "
        f"{len(training_group_index):,} drug-duration strata; "
        f"{len(test_cells)} held-out cells"
    )

    for cell_id in test_cells:
        cell_rows = pert_obs.loc[
            (split_labels == "test")
            & (pert_obs["_cell_id"] == cell_id)
        ].copy()
        require(len(cell_rows) > 0, f"{split_type}/{cell_id}: no test perturbations.")

        test_conditions = (
            cell_rows[["_drug_id", "_dose", "_time"]]
            .drop_duplicates()
            .sort_values(["_drug_id", "_time", "_dose"])
            .reset_index(drop=True)
        )
        require(
            len(test_conditions) > 0,
            f"{split_type}/{cell_id}: no valid unique test conditions.",
        )

        match_mask = matching_training_mask_indexed(
            train_rows,
            training_group_index,
            test_conditions,
        )
        matching_training_rows = int(match_mask.sum())
        exposure_fraction = matching_training_rows / total_training_rows

        require(
            0.0 <= exposure_fraction <= 1.0,
            f"{split_type}/{cell_id}: exposure fraction outside [0,1].",
        )

        exposure_rows.append({
            "split_type": split_type,
            "cell_id": cell_id,
            "condition_training_exposure_fraction": exposure_fraction,
            "n_matching_training_perturbations": matching_training_rows,
            "n_total_training_perturbations": total_training_rows,
            "n_test_perturbations": len(cell_rows),
            "n_unique_test_conditions": len(test_conditions),
            "n_unique_test_drugs": cell_rows["_drug_id"].nunique(),
            "n_test_drug_duration_strata": (
                test_conditions[["_drug_id", "_time"]]
                .drop_duplicates()
                .shape[0]
            ),
            "dose_tolerance_fraction": CONFIG["dose_tolerance_fraction"],
        })

        if CONFIG["save_matched_training_rows_audit"]:
            matched = train_rows.loc[
                match_mask,
                ["index", "_cell_id", "_drug_id", "_dose", "_time"],
            ].copy()
            matched = matched.rename(columns={
                "index": "training_obs_index",
                "_cell_id": "training_cell_id",
                "_drug_id": "drug_id",
                "_dose": "training_dose",
                "_time": "training_duration",
            })
            matched.insert(0, "test_cell_id", cell_id)
            matched.insert(0, "split_type", split_type)
            matched_training_rows_audit.append(matched)

exposure_table = pd.DataFrame(exposure_rows)
training_condition_counts = pd.concat(condition_count_rows, ignore_index=True)

require(
    not exposure_table.duplicated(["split_type", "cell_id"]).any(),
    "Duplicate held-out cell exposure rows detected.",
)
require(
    exposure_table[EXPOSURE_COLUMN].between(0, 1).all(),
    "Condition exposure fractions must lie in [0,1].",
)
require(
    (
        exposure_table["n_matching_training_perturbations"]
        <= exposure_table["n_total_training_perturbations"]
    ).all(),
    "Matched training-row count exceeds total training-row count.",
)

save_table(
    exposure_table,
    "drug_dose_duration_training_exposure_by_test_cell",
)
save_table(
    training_condition_counts,
    "training_condition_counts_by_split",
)

if CONFIG["save_matched_training_rows_audit"] and matched_training_rows_audit:
    matched_training_rows_audit = pd.concat(
        matched_training_rows_audit,
        ignore_index=True,
    )
    save_table(
        matched_training_rows_audit,
        "matched_training_rows_by_test_cell",
    )

display(
    exposure_table.sort_values(
        ["split_type", EXPOSURE_COLUMN]
    )
)


## 6. Repeat the Analysis 1A association framework

In [ ]:
analysis_cell_table = primary_metrics.merge(
    exposure_table,
    on=["split_type", "cell_id"],
    how="left",
    validate="one_to_one",
)

require(
    analysis_cell_table[EXPOSURE_COLUMN].notna().all(),
    f"Missing {EXPOSURE_COLUMN} for at least one held-out cell.",
)

save_table(analysis_cell_table, "training_exposure_cell_level_analysis")

association_rows = []
for metric in METRIC_COLUMNS:
    for split_type in CONFIG["cell_splits"]:
        sub = analysis_cell_table.loc[
            analysis_cell_table["split_type"] == split_type
        ].copy()

        stats = bootstrap_fold_association(
            sub,
            x_col=EXPOSURE_COLUMN,
            y_col=metric,
            n_bootstrap=CONFIG["n_bootstrap"],
            seed=deterministic_seed(
                CONFIG["random_seed"],
                ANALYSIS_NAME,
                split_type,
                metric,
            ),
        )

        association_rows.append({
            "analysis_name": ANALYSIS_NAME,
            "analysis_context": "observed_mixture",
            "exposure_metric": EXPOSURE_COLUMN,
            "metric": metric,
            "split_type": split_type,
            **stats,
        })

fold_associations = pd.DataFrame(association_rows)
save_table(fold_associations, "training_exposure_fold_associations")

across_fold_summary = (
    fold_associations.groupby(["analysis_name", "metric"], as_index=False)
    .agg(
        n_folds=("split_type", "nunique"),
        n_estimable_folds=("association_estimable", "sum"),
        mean_fold_spearman_rho=("spearman_rho", "mean"),
        sd_fold_spearman_rho=("spearman_rho", "std"),
        min_fold_spearman_rho=("spearman_rho", "min"),
        max_fold_spearman_rho=("spearman_rho", "max"),
    )
)
save_table(across_fold_summary, "training_exposure_across_fold_summary")

display(analysis_cell_table.sort_values(["split_type", EXPOSURE_COLUMN]))
display(fold_associations.round(4))
display(across_fold_summary.round(4))

## 7. Export analysis manifest

In [ ]:
manifest = {
    "analysis_name": ANALYSIS_NAME,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "definition": EXPOSURE_DEFINITION,
    "configuration": {
        k: str(v) if isinstance(v, Path) else v
        for k, v in CONFIG.items()
    },
    "unit_of_analysis": "Held-out cell within cell split",
    "association": (
        "Within each split, Spearman correlation across held-out cells between "
        "training exposure fraction and cell-level performance. Percentile "
        "bootstrap 95% confidence intervals resample held-out cells."
    ),
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scanpy": sc.__version__,
    },
}
(MANIFEST_DIR / "analysis_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str)
)
print(f"Completed {ANALYSIS_NAME}. Outputs written to: {OUT}")